# Day 50: Bias-Variance Tradeoff

## The Concept
Bias and variance are two different sources of prediction error, and understanding the tradeoff between them is one of the most fundamental ideas in ML - it explains why models fail and guides how to fix them. Bias is the error from a model being too simple to capture the real underlying pattern - it makes systematic, consistent mistakes regardless of which training data it saw (underfitting). Variance is the error from a model being too sensitive to the specific training data it happened to see - it fits the training set very closely, including its noise, but changes wildly and performs poorly on new data (overfitting). Total expected error is roughly bias^2 + variance + irreducible noise, so reducing one often increases the other (e.g. a simpler model has more bias but less variance, and vice versa), which is why there's a "sweet spot" of model complexity rather than a model that's simply "more complex = always better." The cell below generates a noisy sine-wave dataset and splits it, giving us a clearly non-linear pattern to test different model complexities against.

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split

np.random.seed(0)
X = np.linspace(0, 10, 100).reshape(-1, 1)
y = np.sin(X).ravel() + np.random.normal(0, 0.2, 100)  # smooth sine curve plus noise - a genuinely non-linear pattern
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

## Underfitting Example
`max_depth=1` restricts the decision tree to a single split, so it can only produce two flat prediction regions - nowhere near enough flexibility to trace a sine wave. This is a textbook high-bias model: it makes the same kind of oversimplified mistake everywhere, regardless of which subset of training data it was given. The giveaway for underfitting is that both train and test scores are low and similar to each other - the model isn't doing well even on the data it was trained on, so there's no gap to speak of, just a consistently poor fit. Watch the "train R2" printed below - if it's low, that alone (before even looking at test R2) tells you the model is too simple for this data.

In [ ]:
shallow = DecisionTreeRegressor(max_depth=1).fit(X_train, y_train)  # only 1 split allowed - very little flexibility
print('Shallow tree - train R2:', shallow.score(X_train, y_train), 'test R2:', shallow.score(X_test, y_test))  # both low and close together = underfitting (high bias)

## Overfitting Example
With `max_depth=None`, the tree is allowed to keep splitting until every leaf is pure (or as small as possible), which gives it enough flexibility to essentially memorize the exact value of every training point, including the random noise added on top of the sine wave. This is a textbook high-variance model: it performs almost perfectly on training data but that performance doesn't transfer, because it learned noise-specific quirks rather than the true underlying sine pattern. The telltale sign of overfitting is a large gap between train R2 (very high, often close to 1.0) and test R2 (noticeably lower) - unlike the underfitting case, here the model looks great on paper until you check unseen data.

In [ ]:
deep = DecisionTreeRegressor(max_depth=None).fit(X_train, y_train)  # unrestricted depth - tree can grow until leaves are pure
print('Deep tree - train R2:', deep.score(X_train, y_train), 'test R2:', deep.score(X_test, y_test))  # big gap between train and test = overfitting (high variance)

## Learning Curves
A learning curve plots model performance (here, R2) on both the training set and a cross-validated score as the amount of training data used grows, and it's one of the standard diagnostic tools for telling apart bias and variance problems in practice (more systematic than comparing two hand-picked tree depths). If both curves converge to a low score, the model has high bias - more data won't help much because the model itself is too simple; you need a more flexible model. If there's a persistent large gap between a high training score and a lower validation score that doesn't close as data grows, the model has high variance - collecting more training data, or reducing model complexity/adding regularization, tends to help. Here we use a moderate `max_depth=4` tree, which should show a smaller (and possibly still shrinking) gap than the fully unrestricted tree above, illustrating a decent complexity choice.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve

# for each training-set size, learning_curve does 5-fold CV and returns train/validation scores
train_sizes, train_scores, val_scores = learning_curve(
    DecisionTreeRegressor(max_depth=4), X, y, cv=5, train_sizes=np.linspace(0.1, 1.0, 5)
)
plt.plot(train_sizes, train_scores.mean(axis=1), label='Train score')
plt.plot(train_sizes, val_scores.mean(axis=1), label='Validation score')  # gap between the two curves signals bias vs variance
plt.xlabel('Training set size')
plt.ylabel('R2 Score')
plt.legend()
plt.title('Learning Curve (max_depth=4)')
plt.show()